In [ ]:
# Written by Jayden
import pandas as pd
import numpy as np
import re
import xgboost as xg
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from pathlib import Path

In [2]:
from fractions import Fraction

def fraction_to_decimal(fraction_str):
  """Converts a fraction string to its decimal representation."""
  try:
    return float(Fraction(fraction_str))
  except ValueError:
    return "Invalid fraction format"
  except ZeroDivisionError:
      return "Cannot divide by zero"

In [3]:
file_path = Path("../output/bach_chord_int_data.csv")

try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")

df['duration'] = df['duration'].apply(Fraction)
df['duration'] = df['duration'].astype(float)

display(df.head())

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label,duration,file,position
0,0,0,0,0,0,17,0.500000,"Bach, Johann Sebastian, Partita in B-flat majo...",0
1,0,0,0,0,17,17,0.333333,"Bach, Johann Sebastian, Partita in B-flat majo...",1
2,0,0,0,17,17,17,0.250000,"Bach, Johann Sebastian, Partita in B-flat majo...",2
3,0,0,17,17,17,21,0.250000,"Bach, Johann Sebastian, Partita in B-flat majo...",3
4,0,17,17,17,21,17,0.083333,"Bach, Johann Sebastian, Partita in B-flat majo...",4


In [4]:
X_label = df.drop(columns = ['label', 'duration', 'position', 'file'])
y_label = df['label']

X_duration = df.drop(columns = ['duration', 'position', 'file'])
y_duration = df['duration']

display(X_label.head())
display(y_label.head())
display(X_duration.head())
display(y_duration.head())

X_label_train, X_label_val, y_label_train, y_label_val = train_test_split(X_label, y_label, test_size=0.2, random_state=1)
X_duration_train, X_duration_val, y_duration_train, y_duration_val = train_test_split(X_duration, y_duration, test_size=0.2, random_state=1)

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5
0,0,0,0,0,0
1,0,0,0,0,17
2,0,0,0,17,17
3,0,0,17,17,17
4,0,17,17,17,21


0    17
1    17
2    17
3    21
4    17
Name: label, dtype: int64

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label
0,0,0,0,0,0,17
1,0,0,0,0,17,17
2,0,0,0,17,17,17
3,0,0,17,17,17,21
4,0,17,17,17,21,17


0    0.500000
1    0.333333
2    0.250000
3    0.250000
4    0.083333
Name: duration, dtype: float64

In [5]:
print(y_label.max())

3878


In [6]:
le_label = LabelEncoder()
le_duration = LabelEncoder()

xgb_label_model = xg.XGBClassifier()
xgb_duration_model = xg.XGBClassifier()

In [7]:
'''X_label_train = X_label_train[0:10000]
y_label_train = y_label_train[0:10000]
X_duration_train = X_duration_train[0:50000]
y_duration_train = y_duration_train[0:50000]'''

y_label_train = le_label.fit_transform(y_label_train)
y_duration_train = le_duration.fit_transform(y_duration_train)

In [13]:
#start_time = time.perf_counter()
xgb_label_model.fit(X_label_train, y_label_train)
elapsed_time = time.perf_counter() - start_time
#print(f"Time taken to train label model: {elapsed_time}")

#start_time = time.perf_counter()
xgb_duration_model.fit(X_duration_train, y_duration_train)
elapsed_time = time.perf_counter() - start_time
#print(f"Time taken to train duration model: {elapsed_time}")

In [9]:
y_label_preds = xgb_label_model.predict(X_label_val)
y_label_preds = le_label.inverse_transform(y_label_preds)

In [10]:
accuracy_label = accuracy_score(y_label_val, y_label_preds)
#print(f"Validation Accuracy(label): {accuracy_label}")

In [11]:
lookback1 = 0
lookback2 = 0
lookback3 = 0
lookback4 = 0
lookback5 = 0
position = 0
duration = 0.5

label_data = {'lookback_1' : [lookback1],
              'lookback_2' : [lookback2],
              'lookback_3' : [lookback3],
              'lookback_4' : [lookback4],
              'lookback_5' : [lookback5]}
label_df = pd.DataFrame(label_data)
#print(label_df)

duration_data = {'lookback_1' : [lookback1],
                 'lookback_2' : [lookback2],
                 'lookback_3' : [lookback3],
                 'lookback_4' : [lookback4],
                 'lookback_5' : [lookback5],
                 'label' : [duration]}
duration_df = pd.DataFrame(duration_data)
#print(duration_df)

print("chord, duration")
for i in range(128):
    #label_prediction = xgb_label_model.predict(label_df)
    #label_prediction = le_label.inverse_transform(label_prediction)
    label_probabilities = xgb_label_model.predict_proba(label_df)
    label_prediction = np.array([np.random.choice(xgb_label_model.classes_, p=label_probabilities[0])])
    label_prediction = le_label.inverse_transform(label_prediction)

    for x in range(2, 6):
        label_df[f'lookback_{x-1}'] = label_df[f'lookback_{x}']
    label_df['lookback_5'] = label_prediction
    duration_df['label'] = label_prediction
    #print(label_df)

    duration_prediction = xgb_duration_model.predict(duration_df)
    duration_prediction = le_duration.inverse_transform(duration_prediction)
    #print(duration_df)
    print(f"{label_prediction[0]}, {Fraction(duration_prediction[0]).limit_denominator(12)}")

    for x in range(2, 6):
        duration_df[f'lookback_{x-1}'] = duration_df[f'lookback_{x}']
    duration_df['lookback_5'] = label_prediction

chord, duration
16, 1/4
36, 1/2
36, 1/4
512, 1/4
661, 1/6
144, 1/4
128, 1/12
144, 1/4
128, 1/12
662, 1/12
528, 1/6
516, 1/4
2305, 1/4
5, 1/4
4, 1/4
2324, 1/4
769, 1/4
773, 1/6
2194, 1/4
2176, 1/6
2368, 1/4
2368, 1/12
160, 1/4
32, 1/4
1, 1/4
1, 1/12
2176, 1/4
2176, 1/12
1409, 1/4
1, 1/4
2133, 1/4
145, 1/4
1669, 1/4
656, 1/12
1, 1/6
1, 1/4
513, 1/4
1153, 1/4
16, 1/4
128, 1/4
128, 1/4
132, 1/4
4, 1/4
20, 1/4
5, 1/4
17, 1/4
128, 1/4
49, 1/4
512, 1/4
129, 1/4
161, 1/4
128, 1/4
128, 1/4
128, 1/4
2052, 1/4
2048, 1/12
576, 1/4
1, 1/4
2065, 1/4
512, 1/4
2052, 1/4
2560, 1/4
17, 1/4
21, 1/4
4, 1/6
517, 1/4
513, 1/4
1152, 1/4
544, 1/4
32, 1/4
1060, 1/4
544, 1/4
548, 1/4
516, 1/4
580, 1/4
2084, 1/4
2229, 1/12
128, 1/12
146, 1/4
129, 1/4
4, 1/4
1, 1/12
1, 1/12
1, 1/12
5, 1/4
17, 1/6
1, 1/4
1, 1/4
1, 1/12
1, 1/12
1, 1/12
21, 1/4
516, 1/4
516, 1/4
4, 1/4
5, 1/4
5, 1/12
2324, 1/4
2561, 1/6
2560, 1/4
512, 1/12
1, 1/12
789, 1/4
1, 1/12
1, 1/4
69, 1/4
64, 1/12
2113, 1/12
2049, 1/6
129, 1/4
85, 1/4
129, 1/

In [12]:
print(label_df)
print(duration_df)

   lookback_1  lookback_2  lookback_3  lookback_4  lookback_5
0         129         132         130          16         128
   lookback_1  lookback_2  lookback_3  lookback_4  lookback_5  label
0         129         132         130          16         128    128


In [12]:
print(label_probabilities)

[[1.32850423e-01 1.01674600e-02 9.54825506e-02 1.21863550e-02
  2.15476641e-04 6.60479942e-04 1.61573043e-04 1.03316837e-04
  1.32309055e-04 2.85082951e-05 8.04890238e-04 1.46733103e-02
  3.75315649e-05 1.40206364e-03 1.05262222e-02 4.32657339e-02
  1.58241135e-03 1.23136991e-03 1.30486358e-02 3.10266973e-03
  2.13951641e-03 1.74512446e-04 4.63933211e-05 6.44662753e-02
  2.28824746e-03 1.35087234e-03 3.94175714e-03 4.06992367e-05
  6.12403499e-04 3.73524032e-04 1.27202962e-04 7.53643631e-04
  9.68860986e-04 2.09270991e-04 6.15367957e-04 3.39961494e-04
  4.05592262e-04 1.34543376e-03 1.02310069e-03 7.25993365e-02
  3.24389315e-04 1.27755443e-03 1.07034597e-04 2.60515744e-03
  3.90709320e-04 2.97494320e-04 4.36236587e-05 9.51553229e-03
  1.21295813e-03 2.48102471e-04 5.00470014e-05 1.79052222e-04
  6.54987351e-04 7.92242703e-04 2.04825905e-04 6.56102737e-03
  7.00672914e-04 1.40784250e-04 4.73038584e-04 1.11474493e-03
  4.15246846e-04 3.16434586e-03 6.12227741e-05 1.51516042e-05
  4.9437